# Backbone transfer: CNN-LSTM, TCN, Transformer on Panasonic to CMU eVTOL

This notebook runs `scripts/benchmarks/run_backbone_transfer_cmu.py`, which
reimplements the Vidal transfer protocol (Panasonic 25 C 1 Hz source, nested
chronological target-label budgets 0/1/5/10/40/100%, full-parameter fine-tuning
restarted from the frozen source checkpoint at every budget) for three generic
backbones already registered in `battery_benchmark.models`: `cnn_lstm`, `tcn`,
and `transformer_encoder`. It does not import or modify
`battery_benchmark.reproductions.vidal_2019`; the published Vidal LSTM CMU
results in `docs/benchmarks/vidal_panasonic_to_cmu_evtol_results_v0_1.md` are
unaffected.

Checkpointing is fine-grained: the source checkpoint for each (model, seed) is
saved to Drive as soon as it finishes, and every single (model, seed, cell,
budget) result is written to its own file the moment it finishes -- `cell_metrics.csv`
is rewritten after every one of those, not just once per cell. `--resume`
skips anything already on disk, so a Colab disconnect costs at most the one
budget that was in flight, not a whole cell or backbone.

CMU `soc_dynamic` is a development reference, not laboratory ground truth;
these runs are `formal_benchmark_eligible=false`.

In [ ]:
from pathlib import Path
import os, subprocess, sys, tempfile
from urllib.parse import urlunsplit

from google.colab import drive, userdata
drive.mount('/content/drive')

REPO = Path('/content/battery-soc-transfer-benchmark')
REPO_URL = urlunsplit(('https', 'github.com', '/zhangcastle/battery-soc-transfer-benchmark.git', '', ''))
github_token = userdata.get('GITHUB_TOKEN')
if not github_token:
    raise RuntimeError('Add the GITHUB_TOKEN Colab Secret and enable Notebook access')
if REPO.exists() and not (REPO / '.git').exists():
    raise RuntimeError(f'Remove incomplete checkout and rerun: {REPO}')
with tempfile.TemporaryDirectory() as temp_dir:
    askpass = Path(temp_dir) / 'git-askpass.sh'
    askpass.write_text('#!/bin/sh\ncase "$1" in\n*Username*) echo "x-access-token" ;;\n*) printf "%s" "$GITHUB_TOKEN" ;;\nesac\n')
    askpass.chmod(0o700)
    git_env = os.environ.copy()
    git_env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': github_token})
    command = (['git', '-C', str(REPO), 'pull', '--ff-only'] if (REPO / '.git').exists() else ['git', 'clone', '--depth', '1', REPO_URL, str(REPO)])
    result = subprocess.run(command, env=git_env, text=True, capture_output=True)
del github_token
if result.returncode:
    raise RuntimeError('Authenticated GitHub setup failed; check token permission')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
os.chdir(REPO)
src_path = str(REPO / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
print('Repository ready:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

## Paths (Drive-persisted input/output roots)

Defined once, right after the clone, so every later cell -- including the
smoke test -- uses the real Drive-mounted Panasonic path instead of a
fallback that only exists for people who haven't set up
`configs/paths/colab_drive.yaml`.

In [ ]:
import torch
from battery_benchmark.utils.config import load_yaml

PATHS = load_yaml(REPO / 'configs/paths/colab_drive.yaml')
PANASONIC_ROOT = Path(PATHS['raw_data_root']) / 'Panasonic_18650PF'
CMU_BASE_ROOT = REPO / 'outputs/datasets/cmu_evtol/dynamic_capacity_soc_development_v1'
CMU_CONDITION_ROOT = REPO / 'outputs/datasets/cmu_evtol/dynamic_capacity_soc_development_v1_extension4'
CMU_MANIFEST = REPO / 'data/manifests/cmu_evtol/cmu_evtol_supervised_budget_manifest_v0_1.csv'
RESULT_ROOT = Path(PATHS['output_root']) / 'benchmarks/panasonic_to_cmu_evtol/backbone_transfer'

for path in (PANASONIC_ROOT, CMU_BASE_ROOT, CMU_CONDITION_ROOT, CMU_MANIFEST):
    assert path.exists(), f'Missing required input: {path}'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
print('panasonic root:', PANASONIC_ROOT)
print('results (saved to Drive, survives a disconnect):', RESULT_ROOT)

## Protocol

- Source: Panasonic NCR18650PF, 25 C only, all nine profiles, 1 Hz, 50-step
  non-overlapping windows, train-only min-max normalisation (matches the
  Vidal Case 1 reconstruction assumptions documented in
  `docs/known_uncertainties.md`).
- Target budgets: `zero_shot`, `1pct`, `5pct`, `10pct`, `partial` (~40%), `full`
  (100%) -- the same nested mission-budget flags already frozen in
  `cmu_evtol_supervised_budget_manifest_v0_1.csv` and used by the published
  Vidal LSTM sweep, so budget points line up exactly across backbones.
- Every budget restarts full-parameter fine-tuning from the same frozen
  source checkpoint (Vidal Case 2 pattern); no adapted state carries across
  budgets or cells.
- Model capacity is intentionally left at each backbone's default
  configuration rather than shrunk to match Vidal's 571-parameter LSTM;
  `n_parameters` is recorded per model in every result row and in
  `run_metadata.json` so capacity differences stay visible rather than being
  read as an architecture effect.

## Before the full sweep: a 2-minute smoke test

Run this once after a fresh clone/pull to catch a broken import or a bad
path before committing GPU time to the full sweep. It uses 2 epochs, one
cell, and two budgets, and writes to a throwaway output folder.

In [ ]:
smoke_root = Path('/content/backbone_smoke_test')
subprocess.run(
    [
        sys.executable, '-u', str(REPO / 'scripts/benchmarks/run_backbone_transfer_cmu.py'),
        '--panasonic-data-root', str(PANASONIC_ROOT),
        '--cmu-base-root', str(CMU_BASE_ROOT),
        '--cmu-condition-root', str(CMU_CONDITION_ROOT),
        '--cmu-manifest', str(CMU_MANIFEST),
        '--output-root', str(smoke_root),
        '--models', 'cnn_lstm',
        '--cells', 'VAH01',
        '--budgets', 'zero_shot', '1pct',
        '--source-epochs', '2', '--transfer-epochs', '2',
        '--device', device,
    ],
    check=True,
)
print('smoke test passed')

## Setup for the full sweep

`SEEDS = [42]` by default -- extend to `[42, 43, 44]` for a 3-seed run once
the single-seed sweep looks sane. Multi-seed multiplies the source
pretraining cost by the number of seeds, so it's worth checking the
single-seed curve first.

In [ ]:
import pandas as pd
from IPython.display import display

CELLS = ['VAH01', 'VAH17', 'VAH27', 'VAH30', 'VAH15', 'VAH12', 'VAH11']
BUDGETS = ['zero_shot', '1pct', '5pct', '10pct', 'partial', 'full']
SEEDS = [42]
SOURCE_EPOCHS, TRANSFER_EPOCHS, BATCH_SIZE = 200, 100, 64

def run_backbone(model_name: str) -> None:
    command = [
        sys.executable, '-u', str(REPO / 'scripts/benchmarks/run_backbone_transfer_cmu.py'),
        '--panasonic-data-root', str(PANASONIC_ROOT),
        '--cmu-base-root', str(CMU_BASE_ROOT),
        '--cmu-condition-root', str(CMU_CONDITION_ROOT),
        '--cmu-manifest', str(CMU_MANIFEST),
        '--output-root', str(RESULT_ROOT),
        '--models', model_name,
        '--cells', *CELLS,
        '--budgets', *BUDGETS,
        '--seeds', *[str(s) for s in SEEDS],
        '--source-epochs', str(SOURCE_EPOCHS),
        '--transfer-epochs', str(TRANSFER_EPOCHS),
        '--batch-size', str(BATCH_SIZE),
        '--device', device,
        '--resume',
    ]
    subprocess.run(command, check=True)

## Run: cnn_lstm

Split into one cell per backbone so a disconnect during (say) `tcn` doesn't
require re-touching `cnn_lstm`'s already-finished cells -- with `--resume`
and the fine-grained checkpointing above, re-running a finished backbone's
cell is nearly instant (every budget is skipped).

In [ ]:
run_backbone('cnn_lstm')

## Run: tcn

In [ ]:
run_backbone('tcn')

## Run: transformer_encoder

In [ ]:
run_backbone('transformer_encoder')

## Result 1 - overall MAE / RMSE by model, cell, and budget

In [ ]:
result = pd.read_csv(RESULT_ROOT / 'cell_metrics.csv')
pivot = result.pivot_table(
    index=['cell_id', 'budget'], columns='model',
    values=['overall_mae_percent', 'overall_rmse_percent'],
).round(3)
display(pivot)

mean_by_model_budget = result.groupby(['model', 'budget'])[['overall_mae_percent', 'overall_rmse_percent']].mean().round(3)
display(mean_by_model_budget.rename_axis(['model', 'budget']))

## Result 2 - label efficiency curve (mean MAE across the seven cells, per budget)

Use this to compare against `docs/benchmarks/vidal_panasonic_to_cmu_evtol_results_v0_1.md`,
which reports the same six budget points for the Vidal LSTM (unweighted mean
cell MAE: 35.56 / 8.29 / 6.72 / 5.77 / 5.67 / 2.49 for zero-shot / 1% / 5% /
10% / Partial / Full).

In [ ]:
budget_order = ['zero_shot', '1pct', '5pct', '10pct', 'partial', 'full']
curve = (
    result.groupby(['model', 'budget'])['overall_mae_percent']
    .mean()
    .unstack('model')
    .reindex(budget_order)
    .round(3)
)
display(curve)
curve.plot(marker='o', figsize=(7, 4), ylabel='Mean MAE across 7 cells (%)', xlabel='Target-label budget')